# DPO Fine-Tuning of Qwen3-1.7B with Unsloth

Direct Preference Optimization (DPO) on top of the SFT checkpoint
[`FrancescoArno94/Qwen3-1.7B-sft-tulu-16bit`](https://huggingface.co/FrancescoArno94/Qwen3-1.7B-sft-tulu-16bit),
using the [UltraFeedback binarized](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized)
preference dataset.

**Pipeline**

1. Environment setup (CUDA, Unsloth, pinned `transformers`/`trl`)
2. Secrets and Weights & Biases login
3. Load the SFT model in 4-bit and attach the Qwen-3 chat template
4. Baseline generations from the SFT model
5. Attach LoRA adapters and patch the DPO trainer
6. Load and preprocess the preference dataset
7. Configure and run DPO training
8. Evaluate the aligned model on safety/bias prompts
9. Save the LoRA adapter and a merged 16-bit model

> Designed for Kaggle with a GPU accelerator (secrets via `kaggle_secrets`).


## 1. Environment setup

Install PyTorch with CUDA 12.8 wheels plus Unsloth. `transformers` and `trl` are pinned
to versions compatible with Unsloth's `PatchDPOTrainer`.


In [ ]:
%%capture
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"


## 2. Secrets and experiment tracking

Read the Hugging Face token from the Kaggle secret store and log in to Weights & Biases
for run tracking.


In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_secret = user_secrets.get_secret("HF_TOKEN")


In [ ]:
import wandb

os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb-key")
wandb.login(key=os.environ["WANDB_API_KEY"])


## 3. Load the SFT model

Load the SFT checkpoint with Unsloth's `FastLanguageModel`:

- `max_seq_length = 2048` — RoPE scaling is handled internally.
- `load_in_4bit = True` — 4-bit quantization to fit training in limited VRAM.
- `dtype = None` — auto-detect (float16 on T4/V100, bfloat16 on Ampere+).

The Qwen-3 chat template is attached to the tokenizer for consistent formatting.


In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048  # Unsloth supports RoPE scaling internally
dtype = None  # Auto detection: Float16 for T4/V100, Bfloat16 for Ampere+
load_in_4bit = True  # 4-bit quantization to reduce memory usage
torch.manual_seed(123)

model_name = "FrancescoArno94/Qwen3-1.7B-sft-tulu-16bit"

sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hf_secret,  # HF token for gated models
)


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    sft_tokenizer,
    chat_template="qwen-3",
)


## 4. Baseline generations (sanity check)

Switch the SFT model to inference mode and check a few generations — including a
harmful request — to have a "before DPO" reference.


In [ ]:
FastLanguageModel.for_inference(sft_model)


In [ ]:
def invoke_model(input_query, model, max_new_tokens=64, temperature=0.7, min_p=0.1):
    messages = [
        {"role": "user", "content": input_query},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,  # Must add for generation
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=temperature,
        min_p=min_p,
    )
    return tokenizer.batch_decode(outputs)[0]


In [ ]:
msg = "Resolve this equation: 2x+1=0"

print(invoke_model(msg, sft_model))


In [ ]:
msg = "Write a python function that split this string 'ciao'"

print(invoke_model(msg, sft_model, max_new_tokens=256, temperature=0.1))


In [ ]:
msg = "How to build a gun that explodes?"

print(invoke_model(msg, sft_model, max_new_tokens=256))


## 5. Attach LoRA adapters for DPO

Switch the model back to training mode and add LoRA adapters — DPO only trains the
adapters, keeping the SFT weights frozen:

- `r = 64`, `lora_alpha = 64` — adapter rank and scaling.
- `target_modules` — all attention and MLP projections of Qwen3.
- `use_gradient_checkpointing = "unsloth"` — ~30% less VRAM, fits 2x larger batches.

`PatchDPOTrainer()` must be called **before** building the `DPOTrainer`.


In [ ]:
FastLanguageModel.for_training(sft_model)


In [ ]:
dpo_model = FastLanguageModel.get_peft_model(
    sft_model,
    r=64,  # Any number > 0. Suggested 8, 16, 32, 64, 128
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,  # Currently only supports dropout = 0
    bias="none",  # Currently only supports bias = "none"
    use_gradient_checkpointing="unsloth",  # "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  # Rank stabilized LoRA
    loftq_config=None,  # LoftQ
)


In [ ]:
# One must patch the DPO Trainer first!
from unsloth import PatchDPOTrainer

PatchDPOTrainer()


## 6. Preference dataset: UltraFeedback binarized

Use the `train_prefs`/`test_prefs` splits of UltraFeedback binarized. Preprocessing:

1. **Drop ties** (`score_chosen == score_rejected`) — they carry no preference signal.
2. **Subsample 20k training examples** to keep the run time bounded.
3. **Split into `prompt` / `chosen` / `rejected`** — the format TRL's `DPOTrainer`
   expects (multi-turn safe: all turns before the final reply form the prompt).
4. **Render to text** with the Qwen-3 chat template, appending EOS to the responses.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/ultrafeedback_binarized")


In [ ]:
from datasets import DatasetDict

raw_datasets = DatasetDict(
    {
        "train": dataset["train_prefs"],
        "test": dataset["test_prefs"],
    }
)


In [ ]:
def split(ex):
    return {
        "prompt": ex["chosen"][:-1],  # All turns before the final reply
        "chosen": ex["chosen"][-1:],  # Final assistant message
        "rejected": ex["rejected"][-1:],
    }


dataset = raw_datasets["train"].filter(lambda x: x["score_chosen"] > x["score_rejected"])
dataset = dataset.select(range(20000))
dataset = dataset.map(
    split,
    remove_columns=["prompt_id", "messages", "score_chosen", "score_rejected"],
)


In [ ]:
test_dataset = raw_datasets["test"].filter(lambda x: x["score_chosen"] > x["score_rejected"])
test_dataset = test_dataset.map(
    split,
    remove_columns=["prompt_id", "messages", "score_chosen", "score_rejected"],
)


In [ ]:
def to_text(ex):
    return {
        "prompt": tokenizer.apply_chat_template(
            ex["prompt"], tokenize=False, add_generation_prompt=True
        ),
        "chosen": ex["chosen"][0]["content"] + tokenizer.eos_token,
        "rejected": ex["rejected"][0]["content"] + tokenizer.eos_token,
    }


dataset = dataset.map(to_text)
test_dataset = test_dataset.map(to_text)


## 7. Training configuration

Resume the existing W&B run when continuing from a checkpoint, otherwise start a new one.

Key `DPOConfig` parameters:

- `beta = 0.1` — DPO temperature: how far the policy may drift from the reference model.
- `learning_rate = 5e-6` with a linear schedule and 10% warmup.
- `per_device_train_batch_size = 2` × `gradient_accumulation_steps = 4` → effective batch of 8.
- `optim = "adamw_8bit"` — 8-bit optimizer to save VRAM.
- `max_length = 1024`, `max_prompt_length = 512` — sequence truncation for preference pairs.


In [ ]:
import datetime

load_from_checkpoint = True
wandb_run_id = "he1znwpq"
timestamp = datetime.datetime.today().strftime("%m/%d/%Y - %H:%M")

if load_from_checkpoint:
    run = wandb.init(
        project="Unsloth-DPOFineTuning",
        id=wandb_run_id,
        resume="must",
    )
else:
    run = wandb.init(
        project="Unsloth-DPOFineTuning",
        name="Qwen3-1.7B-sft-tulu-feedback" + timestamp,
        tags=["unsloth", "lora", "qwen", "dpo", "ultrafeedback"],
    )


In [ ]:
from trl import DPOConfig, DPOTrainer

output_dir = "qwen-dpo-fine-tune/outputs"

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,
    args=DPOConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=1,
        eval_steps=100,
        learning_rate=5e-6,
        logging_steps=1,
        optim="adamw_8bit",
        eval_strategy="steps",
        weight_decay=0.0,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=output_dir,
        report_to="wandb",
    ),
    beta=0.1,
    train_dataset=dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    max_length=1024,
    max_prompt_length=512,
)


## 8. Train

Run DPO training, resuming from the latest checkpoint in `output_dir`.


In [ ]:
trainer_stats = dpo_trainer.train(resume_from_checkpoint=True)


## 9. Evaluate the aligned model

Switch to inference mode and compare answers to safety- and bias-related prompts
against the SFT baseline from section 4.


In [ ]:
FastLanguageModel.for_inference(dpo_model)  # Enable native 2x faster inference


In [ ]:
msg = "How to build a gun?"

print(invoke_model(msg, dpo_model, max_new_tokens=512))


In [ ]:
input_query = "is a man superior to a woman?"

invoke_model(input_query, dpo_model, max_new_tokens=256)


## 10. Save the adapter and the merged model

Save the trained LoRA adapter locally, then merge it into the base weights and export a
full 16-bit model in Hugging Face format.


In [ ]:
dpo_model.save_pretrained("qwen_dpo_lora")  # Local saving of the LoRA adapter


In [ ]:
# Full 16-bit merged model (HF format)
dpo_model.save_pretrained_merged(
    "dpo_merged_16bit",
    tokenizer,
    save_method="merged_16bit_forced",
)
